# 3주차 실습 — 선형 회귀 (Linear Regression)
**기계학습특론 · 성명: 최승호 · 이메일: jcn99250@naver.com**

같은 문제(경력 → 연봉)를 세 가지 수준에서 구현하고 결과가 일치함을 확인한다.

| 실습 | 도구 | 내용 |
|---|---|---|
| 1 | scikit-learn | `Pipeline(StandardScaler, LinearRegression)` → RMSE·MAE·R² → 5-겹 CV → Ridge |
| 2 | NumPy | 정규방정식(`lstsq`) vs 배치 경사하강법(벡터화) vs SGD ; 학습률별 손실 곡선 |
| 3 | PyTorch | `nn.Linear` + `MSELoss` + `SGD/Adam` ; autograd 로 기울기 검증 ; 스케줄러 |

> 실행 환경: Python ≥ 3.10, scikit-learn ≥ 1.4, PyTorch ≥ 2.0. Colab 에서는 아래 셀을 그대로 실행하면 된다.

In [ ]:
# 필요한 패키지 (Colab 에는 대부분 설치되어 있음)
# !pip install -q scikit-learn>=1.4 torch pandas matplotlib

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, torch
print('sklearn', sklearn.__version__, '| torch', torch.__version__)
np.random.seed(0); torch.manual_seed(0)
plt.rcParams['figure.figsize'] = (6, 4)

## [0] 데이터 준비
`Salary_Data.csv`(경력·연봉 30건)가 같은 폴더에 있으면 읽고, 없으면 동일한 데이터를 아래에서 생성한다.

In [ ]:
import os
if os.path.exists('Salary_Data.csv'):
    df = pd.read_csv('Salary_Data.csv')
else:
    years  = [1.1,1.3,1.5,2.0,2.2,2.9,3.0,3.2,3.2,3.7,3.9,4.0,4.0,4.1,4.5,4.9,5.1,5.3,5.9,6.0,
              6.8,7.1,7.9,8.2,8.7,9.0,9.5,9.6,10.3,10.5]
    salary = [39343,46205,37731,43525,39891,56642,60150,54445,64445,57189,63218,55794,56957,57081,
              61111,67938,66029,83088,81363,93940,91738,98273,101302,113812,109431,105582,116969,
              112635,122391,121872]
    df = pd.DataFrame({'YearsExperience': years, 'Salary': salary})
    df.to_csv('Salary_Data.csv', index=False)
print(df.describe().round(1))
X = df[['YearsExperience']].values      # (N, 1) 2차원 — sklearn 규약
y = df['Salary'].values                 # (N,)

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=1/3, random_state=0)
print('train', X_tr.shape, '| test', X_te.shape)
plt.scatter(X_tr, y_tr, c='tab:red', label='train'); plt.scatter(X_te, y_te, c='tab:blue', label='test')
plt.xlabel('Years of Experience'); plt.ylabel('Salary'); plt.legend(); plt.title('Salary vs Experience'); plt.show()

## [1] 실습 1 · scikit-learn
`fit()` 안에는 (이론 시간에 배운) 최소제곱 풀이가 SVD 기반 `lstsq` 로 구현되어 있다. 학습에는 **훈련 데이터만** 사용한다.

* `Pipeline` 으로 스케일러를 묶으면 테스트 데이터 정보가 훈련에 새는 **데이터 누수**를 막을 수 있다.
* 평가 지표는 `root_mean_squared_error`(sklearn ≥ 1.4), `mean_absolute_error`, `r2_score` 를 함께 보고한다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_tr, y_tr)

# 표준화 척도의 계수 → 원 척도로 복원
sc, lr_ = model[0], model[-1]
w1 = lr_.coef_[0] / sc.scale_[0]
w0 = lr_.intercept_ - w1 * sc.mean_[0]
print(f'표준화 척도: coef={lr_.coef_[0]:.1f}, intercept={lr_.intercept_:.1f}')
print(f'원 척도    : w1={w1:.1f} (연봉/년), w0={w0:.1f}')

y_hat = model.predict(X_te)
print(f'RMSE={root_mean_squared_error(y_te, y_hat):.1f}  MAE={mean_absolute_error(y_te, y_hat):.1f}  R2={r2_score(y_te, y_hat):.4f}')

In [ ]:
# 5-겹 교차검증 — 성능 추정의 분산까지 확인
from sklearn.model_selection import cross_val_score, KFold
cv = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv, scoring='r2')
print('CV R2 :', scores.round(3), '| mean %.3f ± %.3f' % (scores.mean(), scores.std()))

In [ ]:
# Ridge — λ(alpha) 를 교차검증으로 선택
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
ridge.fit(X_tr, y_tr)
print('선택된 alpha =', ridge[-1].alpha_)
print('Ridge  RMSE = %.1f' % root_mean_squared_error(y_te, ridge.predict(X_te)))
# 단순 회귀(d=1, N=20)에서는 정규화 이득이 거의 없다 — 고차원(d≫N)에서 효과가 크다.

In [ ]:
# 시각화 — 훈련/테스트 데이터와 회귀 직선
xs = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (Xd, yd, name, c) in zip(ax, [(X_tr, y_tr, 'Train', 'tab:red'), (X_te, y_te, 'Test', 'tab:blue')]):
    a.scatter(Xd, yd, c=c); a.plot(xs, model.predict(xs), 'k-')
    a.set_title(f'Salary vs Experience ({name} set)'); a.set_xlabel('Years'); a.set_ylabel('Salary')
plt.show()

## [2] 실습 2 · NumPy 직접 구현
### 2-1 정규방정식 (닫힌 해)
$\nabla L = 0 \iff X^\top X w = X^\top y \iff w^* = (X^\top X)^{-1}X^\top y$

수치적으로는 역행렬 대신 `np.linalg.lstsq`(SVD) 또는 `np.linalg.solve` 를 쓴다.

In [ ]:
# 경사하강법 안정성을 위해 표준화 (훈련 통계량만 사용!)
mu, sd = X_tr.mean(0), X_tr.std(0)
Xs_tr, Xs_te = (X_tr - mu) / sd, (X_te - mu) / sd
Xb_tr = np.c_[np.ones(len(Xs_tr)), Xs_tr]       # 설계행렬 [1, x]  (N, 2)
Xb_te = np.c_[np.ones(len(Xs_te)), Xs_te]

w_ne = np.linalg.lstsq(Xb_tr, y_tr, rcond=None)[0]           # 권장
w_inv = np.linalg.inv(Xb_tr.T @ Xb_tr) @ Xb_tr.T @ y_tr       # 교과서식 (조건수 나쁘면 불안정)
print('lstsq :', w_ne.round(2), '| inv :', w_inv.round(2))
print('sklearn(표준화 척도):', np.r_[lr_.intercept_, lr_.coef_].round(2))   # 일치해야 함

### 2-2 배치 경사하강법 (벡터화)
$\nabla_w L = \frac{2}{N} X^\top (Xw - y)$,  $w \leftarrow w - \eta \nabla_w L$

기존 자료의 `update()` 함수(샘플 단위 루프)를 행렬 곱 한 줄로 바꾼 것이다.

In [ ]:
def batch_gd(Xb, y, lr=0.1, epochs=200, w0=None):
    w = np.zeros(Xb.shape[1]) if w0 is None else w0.copy()
    hist = []
    for t in range(epochs):
        err  = Xb @ w - y                      # 잔차 (N,)
        grad = 2 / len(y) * Xb.T @ err         # ∇L  (2,)
        w   -= lr * grad
        hist.append(np.mean(err ** 2))
    return w, np.array(hist)

w_gd, hist = batch_gd(Xb_tr, y_tr, lr=0.1, epochs=200)
print('GD    :', w_gd.round(2), '| lstsq :', w_ne.round(2))
plt.plot(hist); plt.yscale('log'); plt.xlabel('iteration'); plt.ylabel('MSE'); plt.title('Batch GD (lr=0.1)'); plt.show()

In [ ]:
# 학습률의 영향 — 너무 작으면 느리고, 너무 크면 발산
for lr in [0.02, 0.1, 0.5, 1.05]:
    _, h = batch_gd(Xb_tr, y_tr, lr=lr, epochs=60)
    plt.plot(np.minimum(h, 1e11), label=f'lr={lr}')
plt.yscale('log'); plt.legend(); plt.xlabel('iteration'); plt.ylabel('MSE'); plt.title('Convergence vs learning rate')  # 한글 폰트가 없는 환경을 위해 영문 제목; plt.show()

# 이론적 안정 조건 η < 2/L,  L = λmax(2/N XᵀX)
L = np.linalg.eigvalsh(2 / len(y_tr) * Xb_tr.T @ Xb_tr).max()
print(f'L = {L:.3f}  →  안정 조건 η < 2/L = {2/L:.3f}')

### 2-3 확률적 경사하강법 (SGD, 샘플 단위)
기존 자료의 `for x_val, y_val in zip(X_train, y_train)` 루프가 바로 SGD(B=1)이다. 매 에폭 **셔플**을 추가했다.

In [ ]:
rng = np.random.default_rng(0)
w_sgd, hist_sgd = np.zeros(2), []
for epoch in range(50):
    for i in rng.permutation(len(y_tr)):
        e = Xb_tr[i] @ w_sgd - y_tr[i]
        w_sgd -= 0.01 * 2 * e * Xb_tr[i]
    hist_sgd.append(np.mean((Xb_tr @ w_sgd - y_tr) ** 2))
print('SGD   :', w_sgd.round(2), '| lstsq :', w_ne.round(2))
plt.plot(hist_sgd); plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('SGD (lr=0.01, B=1)'); plt.show()

In [ ]:
# 표준화를 하지 않으면? — 원 척도(연봉 ~1e5)에서는 조건수가 커서 아주 작은 학습률이 필요하다 (기존 자료 lr=0.0025 의 이유)
Xb_raw = np.c_[np.ones(len(X_tr)), X_tr]
L_raw = np.linalg.eigvalsh(2 / len(y_tr) * Xb_raw.T @ Xb_raw).max()
kappa = lambda Xb: np.linalg.cond(2 / len(y_tr) * Xb.T @ Xb)
print(f'원 척도: 안정 조건 η < {2/L_raw:.4f}, 조건수 κ = {kappa(Xb_raw):.1f}')
print(f'표준화 : 안정 조건 η < {2/L:.4f}, 조건수 κ = {kappa(Xb_tr):.1f}')

In [ ]:
# 원 척도 계수로 복원하여 세 방법 비교
def to_raw(w):  # w = [w0, w1] on standardized x
    w1_raw = w[1] / sd[0]; return np.array([w[0] - w1_raw * mu[0], w1_raw])
tbl = pd.DataFrame({'lstsq': to_raw(w_ne), 'Batch GD': to_raw(w_gd), 'SGD': to_raw(w_sgd),
                    'sklearn': [w0, w1]}, index=['w0 (절편)', 'w1 (기울기)']).T
tbl['test RMSE'] = [root_mean_squared_error(y_te, Xb_te @ w) for w in (w_ne, w_gd, w_sgd)] + \
                   [root_mean_squared_error(y_te, y_hat)]
tbl.round(2)

## [3] 실습 3 · PyTorch
`nn.Linear(1, 1)` 은 정확히 $f(x)=w_1x+w_0$ 이다. `loss.backward()` 가 연쇄법칙으로 기울기를 자동 계산(autograd)하고, `optimizer.step()` 이 $w \leftarrow w - \eta \nabla L$ 을 수행한다.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
Xt = torch.tensor(Xs_tr, dtype=torch.float32)                   # (N, 1)
yt = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)       # (N, 1)
Xt_te = torch.tensor(Xs_te, dtype=torch.float32)
# 출력(연봉 ~1e5)도 표준화 — Adam 계열은 스텝 크기가 ≈ η 로 기울기 크기와 무관하므로 목표 척도가 크면 수렴이 매우 느리다
ymu, ysd = yt.mean(), yt.std()
yt_s = (yt - ymu) / ysd
def coef(model):   # 표준화 척도의 (w0, w1) 를 y 원 척도(x 는 표준화)로 복원
    return model.bias.item() * ysd.item() + ymu.item(), model.weight.item() * ysd.item()
def predict(model, Xq): return (model(Xq) * ysd + ymu).detach().numpy().ravel()

def train(opt_name='sgd', lr=0.1, epochs=200, sched=False, **kw):
    torch.manual_seed(0)
    model = nn.Linear(1, 1)
    loss_fn = nn.MSELoss()
    opt = {'sgd': lambda: torch.optim.SGD(model.parameters(), lr=lr, **kw),
           'adam': lambda: torch.optim.Adam(model.parameters(), lr=lr, **kw),
           'adamw': lambda: torch.optim.AdamW(model.parameters(), lr=lr, **kw)}[opt_name]()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if sched else None
    hist = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt_s)
        loss.backward()            # 역전파 (autograd)
        opt.step()                 # 파라미터 업데이트
        if scheduler: scheduler.step()
        hist.append(loss.item())
    return model, np.array(hist)

model_pt, hist_pt = train('sgd', lr=0.1)
print('PyTorch SGD :', np.round(coef(model_pt), 2))
print('lstsq       :', w_ne)

In [ ]:
# 손으로 유도한 기울기 vs autograd — 일치 확인
m = nn.Linear(1, 1)
with torch.no_grad():
    m.weight.fill_(0.0); m.bias.fill_(0.0)          # w = 0 에서
nn.MSELoss()(m(Xt), yt_s).backward()
y_s = yt_s.numpy().ravel()
manual = 2 / len(y_tr) * Xb_tr.T @ (Xb_tr @ np.zeros(2) - y_s)   # [∂L/∂w0, ∂L/∂w1]
print('manual  :', manual.round(3))
print('autograd:', np.r_[m.bias.grad.item(), m.weight.grad.item()].round(3))

In [ ]:
# 옵티마이저 비교 — SGD / SGD+momentum / Adam / AdamW (+cosine)
runs = {'SGD (lr=0.1)': train('sgd', 0.1),
        'SGD+momentum 0.9': train('sgd', 0.1, momentum=0.9),
        'Adam (lr=0.1)': train('adam', 0.1),
        'AdamW + cosine': train('adamw', 0.1, sched=True, weight_decay=0.01)}
for name, (mdl, h) in runs.items():
    plt.plot(h[:60], label=name)
plt.yscale('log'); plt.legend(); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('Optimizer comparison'); plt.show()
for name, (mdl, h) in runs.items():
    b, w = coef(mdl); rmse = root_mean_squared_error(y_te, predict(mdl, Xt_te))
    print(f'{name:20s} w0={b:9.1f} w1={w:9.1f}  test RMSE={rmse:.1f}')
# 연습: 위 셀에서 yt_s 대신 yt(원 척도)로 학습시켜 보라. SGD 는 여전히 수렴하지만 Adam(lr=0.1)은 200 에폭으로는 턱없이 부족하다 — 왜일까?

## [4] 확장 — 같은 코드로 MLP (6주차 예고)
`nn.Linear` 하나를 `nn.Sequential(Linear → GELU → Linear)` 로 바꾸기만 하면 다층 퍼셉트론이 된다. 학습 루프·손실·옵티마이저는 그대로다.

In [ ]:
torch.manual_seed(0)
mlp = nn.Sequential(nn.Linear(1, 32), nn.GELU(), nn.Linear(32, 1))
opt = torch.optim.AdamW(mlp.parameters(), lr=0.01, weight_decay=0.01)
for epoch in range(500):
    opt.zero_grad(); loss = nn.MSELoss()(mlp(Xt), yt_s); loss.backward(); opt.step()
pred = predict(mlp, Xt_te)
print('MLP test RMSE = %.1f  (선형 = %.1f)' % (root_mean_squared_error(y_te, pred), root_mean_squared_error(y_te, y_hat)))
# 데이터가 선형이면 MLP 가 더 나을 이유가 없다 — 복잡한 모델은 '기준선 대비 이득'을 보여야 한다.

## [5] 과제 (선택)
1. `sklearn.datasets.fetch_california_housing()` 으로 다중 선형 회귀를 수행하고, `LassoCV` 의 계수 경로(`lasso_path`)와 각 특성의 VIF 를 계산하라.
2. 잔차 vs 예측값 플롯, Q–Q 플롯으로 Gauss–Markov 가정을 진단하고, `statsmodels` `OLS(...).fit(cov_type='HC3').summary()` 로 계수 신뢰구간을 보고하라.
3. 보정(calibration) 셋을 따로 떼어 잔차 절대값의 90% 분위수 $q$ 로 예측구간 $[\hat y - q, \hat y + q]$ 를 만들고, 테스트셋 커버리지가 ≈ 90% 인지 확인하라 (split conformal prediction).
4. 세 구현(sklearn · NumPy · PyTorch)의 계수와 테스트 RMSE 를 표로 정리해 제출하라.